# D254 — Olist Hive Query Optimization: Student Lab

Investigate Hive performance using execution plans, columnar storage, partitions, statistics, predicate pruning, join strategy, and file layout. This replaces the MySQL index-tuning exercise: Hive is a distributed analytical engine and is not optimized with ordinary row-store B-tree indexes.


## Lab conventions

- Run this notebook from Jupyter in Ubuntu/WSL with HDFS, YARN, the Hive metastore, and HiveServer2 running.
- Source CSVs are expected at `/mnt/c/data/olist`. Change `OLIST_SOURCE` if your copy is elsewhere.
- Use database `olist_hive`. Raw external tables should be named after the MySQL tables: `olist_customers`, `olist_orders`, `olist_order_items`, `olist_order_payments`, `olist_order_reviews`, `olist_products`, `olist_sellers`, `olist_geolocation`, and `product_category_translation`.
- These are learner exercises. TODO cells intentionally contain no completed answer.


## MySQL → HiveSQL reminders

| MySQL habit | HiveSQL approach in this lab |
|---|---|
| `LIMIT offset, count` | use `LIMIT count`; use `ROW_NUMBER()` when paging is genuinely required |
| `IFNULL(x, y)` | prefer standard `COALESCE(x, y)` |
| `DATEDIFF(end, start)` | same two-argument order in Hive; result is whole days |
| `DATE_FORMAT(ts, '%Y-%m')` | use `date_format(ts, 'yyyy-MM')` |
| `YEAR(ts)` / `MONTH(ts)` | `year(ts)` / `month(ts)` are available |
| `GROUP_CONCAT` | use `concat_ws(',', collect_list(...))` when appropriate |
| backtick-heavy identifiers | use simple lowercase snake_case names; backticks are rarely needed here |
| primary keys, foreign keys, and row indexes | Hive generally does not enforce relational keys; performance comes from file format, partitions, statistics, pruning, and execution settings |
| `EXPLAIN ANALYZE` | use `EXPLAIN`, `EXPLAIN FORMATTED`, and runtime/Tez or YARN evidence available in your installation |

Hive aliases usually cannot be reused by another expression in the same `SELECT`. Put the first calculation in a CTE/subquery. Always make window ordering deterministic by adding an ID as a tie-breaker.


## Safety and evidence rules

- Work only in database `olist_hive_opt` and HDFS paths below `/user/hive/external/d254_olist`.
- Do not overwrite the D250 raw data.
- Capture baseline and changed plans before drawing a conclusion.
- Record row counts and checksums/aggregates to prove optimization did not change results.
- Runtime from a single warm or cold run is weak evidence. Run comparable trials and note cache, queue, and cluster conditions.
- Do not claim a partition or format is faster without plan or runtime evidence.


## 0. Check services and execution engine


In [ ]:
%%bash
jps
ss -lnt | grep -E ':(9083|10000)\b' || true
hdfs dfsadmin -report | grep -E 'Live datanodes|Name:'
yarn node -list


In [ ]:
%%bash
# TODO: Query relevant Hive settings: execution engine, CBO, vectorization, dynamic partition mode, and auto-convert join settings.
# beeline -u 'jdbc:hive2://localhost:10000/olist_hive' -n "$USER" --silent=true -e "
#   -- your HiveQL here
# "


## Exercise 1 — Establish a text-table baseline

Choose a selective order-date query and an item-revenue aggregation against raw text external tables. Save `EXPLAIN FORMATTED`, returned aggregates, and repeated runtimes.

**Tip:** Use the same SQL and output checks for every later comparison.


In [ ]:
%%bash
# TODO: Write and run your solution here.
# beeline -u 'jdbc:hive2://localhost:10000/olist_hive' -n "$USER" --silent=true -e "
#   -- your HiveQL here
# "


## Exercise 2 — Compare TextFile with Parquet or ORC

Create a columnar copy of the required tables with CTAS, rerun the baseline queries, and compare plan, bytes read if available, and runtime.

**Tip:** Columnar storage helps most when queries read a subset of columns and can use predicate/statistics pruning.


In [ ]:
%%bash
# TODO: Write and run your solution here.
# beeline -u 'jdbc:hive2://localhost:10000/olist_hive' -n "$USER" --silent=true -e "
#   -- your HiveQL here
# "


## Exercise 3 — Design an order-date partition

Create a partitioned orders table using a low-to-moderate-cardinality date derivative such as purchase year and month. Populate it, inspect HDFS layout, and compare a filtered plan with an unfiltered plan.

**Tip:** The partition columns must appear in predicates for partition pruning. Do not partition by `order_id`.


In [ ]:
%%bash
# TODO: Write and run your solution here.
# beeline -u 'jdbc:hive2://localhost:10000/olist_hive' -n "$USER" --silent=true -e "
#   -- your HiveQL here
# "


## Exercise 4 — Prove partition pruning

Use `EXPLAIN FORMATTED` on two logically comparable filters: one directly constraining partition columns and one applying a function only to the original timestamp. Identify partitions scanned.

**Tip:** A function on a timestamp is not automatically equivalent to an explicit partition predicate; include both if needed.


In [ ]:
%%bash
# TODO: Write and run your solution here.
# beeline -u 'jdbc:hive2://localhost:10000/olist_hive' -n "$USER" --silent=true -e "
#   -- your HiveQL here
# "


## Exercise 5 — Collect and inspect statistics

Run `ANALYZE TABLE ... COMPUTE STATISTICS` and relevant column-statistics commands on selected optimized tables. Inspect `DESCRIBE FORMATTED` before and after, then compare plans.

**Tip:** Statistics help the cost-based optimizer estimate cardinality and choose join order/strategy; they do not change query results.


In [ ]:
%%bash
# TODO: Write and run your solution here.
# beeline -u 'jdbc:hive2://localhost:10000/olist_hive' -n "$USER" --silent=true -e "
#   -- your HiveQL here
# "


## Exercise 6 — Map-side/broadcast join candidate

Join a large fact table to the small category-translation table. Compare plans with the environment's automatic join conversion enabled and disabled, without forcing unsafe memory use.

**Tip:** A small dimension may be broadcast, but prove the selected join from plan evidence.


In [ ]:
%%bash
# TODO: Write and run your solution here.
# beeline -u 'jdbc:hive2://localhost:10000/olist_hive' -n "$USER" --silent=true -e "
#   -- your HiveQL here
# "


## Exercise 7 — Early aggregation before joins

Compare a query that joins raw payment and item rows before aggregation with a correct version that aggregates each to order grain first. Evaluate both correctness and work performed.

**Tip:** The raw join can multiply rows and produce a wrong answer; performance tuning never excuses incorrect grain.


In [ ]:
%%bash
# TODO: Write and run your solution here.
# beeline -u 'jdbc:hive2://localhost:10000/olist_hive' -n "$USER" --silent=true -e "
#   -- your HiveQL here
# "


## Exercise 8 — Predicate and column pruning

Compare `SELECT *` with a query selecting only needed columns, and compare late filtering with a filter that can be pushed near the scan. Use columnar tables and plan evidence.

**Tip:** Inspect projected columns and filter expressions in the table-scan operator.


In [ ]:
%%bash
# TODO: Write and run your solution here.
# beeline -u 'jdbc:hive2://localhost:10000/olist_hive' -n "$USER" --silent=true -e "
#   -- your HiveQL here
# "


## Exercise 9 — Small-files investigation

Inspect file counts and sizes for one table or partition. Create a controlled small-file version, compare planning/runtime overhead, and propose compaction using CTAS or `INSERT OVERWRITE` into a suitable file layout.

**Tip:** Never run broad HDFS deletion. Keep the experiment under the exact D254 path.


In [ ]:
%%bash
# TODO: Write and run your solution here.
# beeline -u 'jdbc:hive2://localhost:10000/olist_hive' -n "$USER" --silent=true -e "
#   -- your HiveQL here
# "


## Exercise 10 — Window-query shuffle analysis

Explain the plan for a seller leaderboard using partitioned ranking. Identify aggregation, exchange/shuffle, sort, and window stages. Test whether pre-aggregation reduces work.

**Tip:** Hive cannot avoid the logical sort required by ranking merely because a MySQL index might have supported an order.


In [ ]:
%%bash
# TODO: Write and run your solution here.
# beeline -u 'jdbc:hive2://localhost:10000/olist_hive' -n "$USER" --silent=true -e "
#   -- your HiveQL here
# "


## Exercise 11 — Bucketing design (optional)

Propose and, if supported by the course environment, test bucketed copies for a frequently joined high-cardinality key. Verify the physical files and plan rather than assuming the optimizer uses buckets.

**Tip:** Bucketing is not partitioning and benefits depend on correct writes and engine settings.


In [ ]:
%%bash
# TODO: Write and run your solution here.
# beeline -u 'jdbc:hive2://localhost:10000/olist_hive' -n "$USER" --silent=true -e "
#   -- your HiveQL here
# "


## Exercise 12 — Optimization report

Recommend a storage design for recurring Olist analytics. Include format, compression, partition keys, statistics maintenance, join-grain rules, and monitoring evidence. State costs and cases where each choice does not help.

**Tip:** Keep recommendations tied to observed workload, not generic slogans.


In [ ]:
%%bash
# TODO: Write and run your solution here.
# beeline -u 'jdbc:hive2://localhost:10000/olist_hive' -n "$USER" --silent=true -e "
#   -- your HiveQL here
# "


## Required comparison table

Complete this table in your submission:

| Experiment | Same result verified? | Tables/partitions scanned | Join/aggregation strategy | Files or bytes read | Median runtime | Conclusion |
|---|---|---|---|---:|---:|---|
| Text baseline | | | | | | |
| Columnar | | | | | | |
| Partition-pruned | | | | | | |
| With statistics | | | | | | |
| Early aggregation | | | | | | |

If your Hive installation does not expose a metric, write `not available` rather than inventing it.


In [ ]:
# TODO: Add your completed comparison table in a Markdown cell and attach plan evidence.
